# FloodNet preprocessing

This notebook prepares half-resolution image/mask copies for segmentation experiments. It keeps the official `train`, `val`, and `test` assignments, never modifies source files, logs unreadable or unmatched files, and exposes sampling utilities for training and whole-image evaluation.

Run the cells in order. The resize step writes to `Project_3/Data/FloodNet/processed_half`; set `RUN_RESIZE = True` in the execution cell to create the copies. The default mask source is the dataset's `ColorMasks-FloodNetv1.0` tree, with a fallback to the labels shipped beside the original images. No imputation is performed because pixel data are treated as complete; decode failures are excluded and recorded.

In [ ]:
from pathlib import Path
from collections import defaultdict
import csv
import logging
import random

import numpy as np
from PIL import Image, ImageEnhance

SEED = 4002
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Scripts" and PROJECT_ROOT.parent.name == "Project_3":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name != "Project_3":
    PROJECT_ROOT = PROJECT_ROOT / "Project_3"
DATA_ROOT = PROJECT_ROOT / "Data" / "FloodNet"
IMAGE_ROOT = DATA_ROOT / "FloodNet-Supervised_v1.0"
COLOR_MASK_ROOT = DATA_ROOT / "ColorMasks-FloodNetv1.0"
OUTPUT_ROOT = DATA_ROOT / "processed_half"
LOG_ROOT = PROJECT_ROOT / "Scripts" / "preprocessing_logs"
LOG_ROOT.mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
Image.MAX_IMAGE_PIXELS = None


## Build official split pairs and resize copies

Masks are matched by image stem (`1234.jpg` ↔ `1234_lab.png`). The external color-mask tree is preferred when it contains the corresponding mask; otherwise the official split's label image is used. A pair is eligible only if both files decode successfully. Any failure is written to a CSV log.

In [ ]:
SPLITS = {
    "train": {"image_dir": "train/train-org-img", "label_dir": "train/train-label-img", "color_dir": "ColorMasks-TrainSet"},
    "val":   {"image_dir": "val/val-org-img",     "label_dir": "val/val-label-img",     "color_dir": "ColorMasks-ValSet"},
    "test":  {"image_dir": "test/test-org-img",   "label_dir": "test/test-label-img",   "color_dir": "ColorMasks-TestSet"},
}

def mask_candidates(split, image_path):
    stem = image_path.stem
    return [COLOR_MASK_ROOT / SPLITS[split]["color_dir"] / f"{stem}_lab.png",
            IMAGE_ROOT / SPLITS[split]["label_dir"] / f"{stem}_lab.png",
            IMAGE_ROOT / SPLITS[split]["label_dir"] / f"{stem}.png"]

def discover_pairs():
    pairs, issues = [], []
    for split, paths in SPLITS.items():
        for image_path in sorted((IMAGE_ROOT / paths["image_dir"]).glob("*")):
            if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}:
                continue
            mask_path = next((p for p in mask_candidates(split, image_path) if p.is_file()), None)
            if mask_path is None:
                issues.append({"split": split, "image": str(image_path), "mask": "", "reason": "mask not found"})
                continue
            try:
                with Image.open(image_path) as im: im.verify()
                with Image.open(mask_path) as im: im.verify()
            except Exception as exc:
                issues.append({"split": split, "image": str(image_path), "mask": str(mask_path), "reason": f"unreadable: {exc}"})
                continue
            pairs.append({"split": split, "image": image_path, "mask": mask_path, "id": image_path.stem})
    return pairs, issues

def half_size(size):
    return max(1, size[0] // 2), max(1, size[1] // 2)

def write_half_resolution(pairs, issues):
    for row in pairs:
        out_image = OUTPUT_ROOT / row["split"] / "images" / f"{row['id']}.jpg"
        out_mask = OUTPUT_ROOT / row["split"] / "masks" / f"{row['id']}_lab.png"
        out_image.parent.mkdir(parents=True, exist_ok=True)
        out_mask.parent.mkdir(parents=True, exist_ok=True)
        try:
            with Image.open(row["image"]) as im:
                im = im.convert("RGB")
                im.resize(half_size(im.size), Image.Resampling.BILINEAR).save(out_image, quality=95)
            with Image.open(row["mask"]) as im:
                im.resize(half_size(im.size), Image.Resampling.NEAREST).save(out_mask)
        except Exception as exc:
            issues.append({"split": row["split"], "image": str(row["image"]), "mask": str(row["mask"]), "reason": f"resize failure: {exc}"})
            for path in (out_image, out_mask):
                if path.exists(): path.unlink()
    log_path = LOG_ROOT / "unreadable_or_unmatched.csv"
    with log_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["split", "image", "mask", "reason"])
        writer.writeheader(); writer.writerows(issues)
    return log_path

pairs, file_issues = discover_pairs()
print(f"Discovered {len(pairs)} readable image/mask pairs; initial issues: {len(file_issues)}")


In [ ]:
# Set True to write copies; source files are opened read-only and never overwritten.
RUN_RESIZE = False
if RUN_RESIZE:
    log_path = write_half_resolution(pairs, file_issues)
    print(f"Wrote resized data to {OUTPUT_ROOT}; issue log: {log_path}")
else:
    print("Resize disabled. Set RUN_RESIZE = True and rerun this cell to create the copies.")


## Training tiles, augmentation, and normalization

Training samples are random 512 × 512 paired crops from the half-resolution train copies. Horizontal/vertical flips, rotations by multiples of 90°, and brightness jitter apply only to training. ImageNet channel statistics are applied to image inputs; masks are returned unchanged.

In [ ]:
TILE_SIZE = 512
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def normalize_image(image):
    """Return ImageNet-normalized float32 HWC RGB data."""
    arr = np.asarray(image.convert("RGB") if isinstance(image, Image.Image) else image)
    arr = arr.astype(np.float32) / 255.0
    return (arr - IMAGENET_MEAN) / IMAGENET_STD

def random_training_tile(image, mask, tile_size=TILE_SIZE, rng=None, brightness_range=(0.8, 1.2)):
    """Return an augmented normalized image crop and matching unmodified-label crop."""
    rng = rng or random
    image = image.convert("RGB") if isinstance(image, Image.Image) else Image.fromarray(np.asarray(image).astype(np.uint8))
    mask = mask if isinstance(mask, Image.Image) else Image.fromarray(np.asarray(mask))
    if image.size != mask.size: raise ValueError(f"Image/mask size mismatch: {image.size} vs {mask.size}")
    width, height = image.size
    if width < tile_size or height < tile_size: raise ValueError(f"Image {image.size} is smaller than tile {tile_size}")
    left, top = rng.randint(0, width - tile_size), rng.randint(0, height - tile_size)
    box = (left, top, left + tile_size, top + tile_size)
    image, mask = image.crop(box), mask.crop(box)
    if rng.random() < 0.5:
        image, mask = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT), mask.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    if rng.random() < 0.5:
        image, mask = image.transpose(Image.Transpose.FLIP_TOP_BOTTOM), mask.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
    k = rng.randrange(4)
    if k: image, mask = image.rotate(90 * k), mask.rotate(90 * k)
    image = ImageEnhance.Brightness(image).enhance(rng.uniform(*brightness_range))
    return normalize_image(image), np.asarray(mask)


## Whole-image evaluation windows

Evaluation uses deterministic overlapping windows over each half-resolution image. The final window on each axis is anchored to the far edge so every pixel is covered. If an image is smaller than the window, it is padded on the bottom/right. No augmentation is applied.

In [ ]:
def sliding_window_positions(length, window=TILE_SIZE, stride=384):
    if window <= 0 or stride <= 0: raise ValueError("window and stride must be positive")
    if length <= window: return [0]
    positions = list(range(0, length - window + 1, stride))
    last = length - window
    if positions[-1] != last: positions.append(last)
    return positions

def evaluation_windows(image, mask=None, window=TILE_SIZE, stride=384):
    """Yield top, left, normalized image crop, optional raw mask crop in row-major order."""
    image = image.convert("RGB") if isinstance(image, Image.Image) else Image.fromarray(np.asarray(image).astype(np.uint8))
    if mask is not None and not isinstance(mask, Image.Image): mask = Image.fromarray(np.asarray(mask))
    width, height = image.size
    pad_w, pad_h = max(0, window - width), max(0, window - height)
    if pad_w or pad_h:
        arr = np.asarray(image)
        arr = np.pad(arr, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
        image = Image.fromarray(arr.astype(np.uint8))
        if mask is not None:
            m = np.asarray(mask)
            pads = ((0, pad_h), (0, pad_w)) if m.ndim == 2 else ((0, pad_h), (0, pad_w), (0, 0))
            mask = Image.fromarray(np.pad(m, pads, mode="constant", constant_values=0))
        width, height = image.size
    for top in sliding_window_positions(height, window, stride):
        for left in sliding_window_positions(width, window, stride):
            box = (left, top, left + window, top + window)
            crop_mask = mask.crop(box) if mask is not None else None
            yield top, left, normalize_image(image.crop(box)), (np.asarray(crop_mask) if crop_mask is not None else None)


## Test near-duplicate audit

A deterministic 64-bit difference hash flags test images that resemble any train or validation image. The Hamming-distance threshold is configurable. Each test image's closest candidate is recorded; split membership is never changed. This is a screening heuristic, so review flagged rows.

In [ ]:
def difference_hash(image, hash_size=8):
    gray = image.convert("L").resize((hash_size + 1, hash_size), Image.Resampling.BILINEAR)
    pixels = np.asarray(gray, dtype=np.int16)
    value = 0
    for bit in (pixels[:, 1:] > pixels[:, :-1]).flatten(): value = (value << 1) | int(bit)
    return value

def hamming_distance(a, b): return (a ^ b).bit_count()

def audit_test_near_duplicates(pairs, max_distance=8):
    hashes = defaultdict(list)
    for row in pairs:
        try:
            with Image.open(row["image"]) as im: hashes[row["split"]].append((row["id"], difference_hash(im)))
        except Exception as exc: logging.warning("Skipping unreadable image %s: %s", row["image"], exc)
    results = []
    for test_id, test_hash in hashes["test"]:
        candidates = [(hamming_distance(test_hash, ref_hash), ref_id, split)
                      for split in ("train", "val") for ref_id, ref_hash in hashes[split]]
        if not candidates:
            results.append({"test_id": test_id, "flagged": False, "closest_id": "", "closest_split": "", "distance": ""})
            continue
        distance, ref_id, split = min(candidates)
        results.append({"test_id": test_id, "flagged": distance <= max_distance,
                        "closest_id": ref_id, "closest_split": split, "distance": distance})
    return results

PHASH_MAX_DISTANCE = 8
# Audit at original resolution; the report is saved separately from the official data.
duplicate_rows = audit_test_near_duplicates(pairs, PHASH_MAX_DISTANCE)
duplicate_path = LOG_ROOT / "test_near_duplicates.csv"
with duplicate_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["test_id", "flagged", "closest_id", "closest_split", "distance"])
    writer.writeheader(); writer.writerows(duplicate_rows)
print(f"{sum(bool(r['flagged']) for r in duplicate_rows)} test images flagged; report: {duplicate_path}")


## Reproducibility and notes

- Python `random` and NumPy use seed **4002**. For repeatable sample order, pass `random.Random(4002)` to `random_training_tile`.
- Images use bilinear resizing; masks use nearest-neighbor resizing.
- Pixel values are not imputed. Missing pairs and decode failures are excluded and logged.
- Training augmentation is applied only in `random_training_tile`; evaluation windows are deterministic and unaugmented.